# Fundamentos de Deep Learning: de los datos a una predicción

En este notebook construiremos, paso a paso, la intuición que necesitaremos para estudiar redes neuronales.

```text
Pregunta científica
        ↓
Observaciones
        ↓
Features (X) y Labels (y)
        ↓
Modelo
        ↓
Predicción
```

## Objetivos

Al finalizar podrás:

- distinguir entre **sample**, **feature** y **label**;
- interpretar `X.shape` y `y.shape`;
- visualizar relaciones entre variables científicas;
- entender por qué algunas features son más informativas que otras;
- construir una regla de clasificación muy sencilla;
- conectar esta idea con lo que más adelante hará una red neuronal.


## 1. Una pregunta científica

Trabajaremos con el dataset **Iris**, que contiene mediciones físicas de flores de tres especies:

- *Iris setosa*
- *Iris versicolor*
- *Iris virginica*

Nuestra pregunta será:

> **¿Podemos utilizar mediciones de sépalos y pétalos para distinguir especies de flores?**


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris


In [ ]:
iris = load_iris()

df = pd.DataFrame(iris.data, columns=iris.feature_names)
df["species_id"] = iris.target
df["species"] = df["species_id"].map(dict(enumerate(iris.target_names)))

df.head()


Cada **fila** representa una flor observada.

Cada columna de mediciones representa una **feature**.

La especie conocida de la flor será nuestra **label**.

<details>
<summary><strong>Pista: ¿cómo reconocer una feature?</strong></summary>

Piensa en una feature como una **medición que puede realizarse antes de conocer la respuesta**.

</details>


### Pregunta 1

¿Cuántas features tiene cada flor?

Ejecuta la siguiente celda.


In [ ]:
X = iris.data
y = iris.target

print("X.shape =", X.shape)
print("y.shape =", y.shape)


Interpreta:

```text
X.shape = (n_samples, n_features)
```

<details>
<summary><strong> Mostrar solución</strong></summary>

`X.shape = (150, 4)`

- **150 samples**
- **4 features** por flor

`y.shape = (150,)` indica que tenemos una label para cada flor.

</details>


## 2. Observemos un sample


In [ ]:
sample_id = 12

print("Sample:", sample_id)
for feature, value in zip(iris.feature_names, X[sample_id]):
    print(f"{feature:22s}: {value:.2f} cm")

print("Label:", iris.target_names[y[sample_id]])


Podemos imaginar:

```text
4 mediciones físicas
        ↓
      modelo
        ↓
      especie
```

Más adelante, esas cuatro mediciones podrán convertirse en **cuatro entradas de una red neuronal**.


## 3. ¿Todas las features contienen información útil?

Veamos el **largo del pétalo**.


In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(df["petal length (cm)"], bins=15, edgecolor="black")
plt.xlabel("Largo del pétalo (cm)")
plt.ylabel("Número de flores")
plt.title("Distribución del largo del pétalo")
plt.show()


### Pregunta 2

¿Por qué podría ser importante que una feature varíe entre observaciones?

<details>
<summary><strong> Pista</strong></summary>

Si una variable tuviera exactamente el mismo valor para todas las flores, no ayudaría a distinguirlas.

</details>

<details>
<summary><strong> Mostrar solución</strong></summary>

Una feature puede ser útil cuando su variación contiene información relacionada con la clase o cantidad que queremos predecir.

</details>


## 4. Visualizando dos features

Ahora observaremos:

- largo del pétalo;
- ancho del pétalo.


In [ ]:
plt.figure(figsize=(7, 5))

for class_id, species in enumerate(iris.target_names):
    mask = y == class_id
    plt.scatter(X[mask, 2], X[mask, 3], label=species, alpha=0.75)

plt.xlabel("Largo del pétalo (cm)")
plt.ylabel("Ancho del pétalo (cm)")
plt.title("Dos features del dataset Iris")
plt.legend()
plt.show()


### Pregunta 3

1. ¿Qué especie parece más fácil de separar?
2. ¿Qué especies presentan mayor solapamiento?

<details>
<summary><strong> Pista</strong></summary>

Busca grupos de puntos que ocupen regiones diferentes del gráfico.

</details>

<details>
<summary><strong> Mostrar solución</strong></summary>

- **Setosa** aparece claramente separada.
- **Versicolor** y **Virginica** presentan más solapamiento.

Esto ilustra una idea central: **aprender patrones en el espacio de features**.

</details>


## 5. Construyamos una regla de clasificación a mano

Intentemos separar *Setosa* usando solo `petal length`.

Cambia el valor de `threshold` y observa la accuracy.


In [ ]:
threshold = 2.0

pred_setosa = df["petal length (cm)"] < threshold
true_setosa = df["species"] == "setosa"

accuracy = (pred_setosa == true_setosa).mean()

print(f"Threshold: {threshold:.2f} cm")
print(f"Accuracy: {accuracy:.3f}")


Prueba varios valores.

<details>
<summary><strong> Pista</strong></summary>

Inspecciona:

```python
df.groupby("species")["petal length (cm)"].agg(["min", "max"])
```

</details>

<details>
<summary><strong> Mostrar solución</strong></summary>

Un valor alrededor de **1.9–2.0 cm** separa muy bien a Setosa en este dataset.

La idea importante es que una decisión puede construirse a partir de una **feature** y un **parámetro ajustable**.

</details>


In [ ]:
df.groupby("species")["petal length (cm)"].agg(["min", "max", "mean"])


## 6. De una regla manual a una neurona

Una neurona muy sencilla calcula:

\[
z = wx + b
\]

donde:

- \(x\): feature;
- \(w\): weight;
- \(b\): bias.

Probemos una combinación.


In [ ]:
x = 1.4
w = -2.0
b = 4.0

z = w*x + b
prediction = 1 if z > 0 else 0

print("z =", z)
print("prediction =", prediction)


### Pregunta 4

¿Qué ocurre si cambias `x` de `1.4` a `4.5`?

<details>
<summary><strong> Pista</strong></summary>

Calcula:

\[
z=(-2)(4.5)+4
\]

</details>

<details>
<summary><strong> Mostrar solución</strong></summary>

\[
z=-5
\]

La salida cambia de clase.

Esto muestra cómo **feature + weight + bias** pueden producir una decisión.

</details>


## 7. ¿Dónde está el aprendizaje?

Hasta ahora **nosotros** elegimos los parámetros.

Una red neuronal aprende ajustándolos:

```text
Datos
  ↓
Predicción
  ↓
Comparar con la respuesta correcta
  ↓
Calcular error
  ↓
Modificar weights y biases
  ↓
Repetir
```

Más adelante veremos:

```text
Forward Pass → Loss → Backpropagation → Update
```

> **Aprender significa ajustar parámetros para reducir errores.**


## 8. Conexión con ciencia

### Meteorología

```text
Temperatura ──┐
Humedad ──────┼──► modelo ───► lluvia / no lluvia
Presión ──────┤
Viento ───────┘
```

### Astronomía

```text
Temperatura ──┐
Luminosidad ──┼──► modelo ───► tipo de estrella
Color ────────┤
Radio ────────┘
```

### Física de partículas

```text
Energía ──────────────┐
Momento ──────────────┼──► modelo ───► tipo de evento
Dirección ────────────┤
Señales del detector ─┘
```


# Reto rápido

Tenemos:

```text
temperature
humidity
pressure
wind_speed
station_id
rain
```

¿Cuáles usarías como features y cuál sería la label?

<details>
<summary><strong> Pista</strong></summary>

Busca las **mediciones físicas relevantes**.

</details>

<details>
<summary><strong> Mostrar solución</strong></summary>

**Features**

- temperature
- humidity
- pressure
- wind_speed

**Label**

- rain

`station_id` es un identificador, no una medición física del estado atmosférico.

</details>


# Para recordar

\[
X = \text{features},
\qquad
y = \text{labels}
\]

y:

```text
Pregunta científica
        ↓
Mediciones
        ↓
Features y Labels
        ↓
Parámetros
        ↓
Predicción
        ↓
Aprendizaje
```

En el próximo módulo estudiaremos cómo una **neurona artificial** combina múltiples features.
